### Step 1: Install Dependencies
Run this cell once. Installs all packages and **auto-restarts the kernel**.
Running it a second time skips straight through.

In [ ]:
import sys, os

if not os.path.exists('/kaggle/working/OOTDiffusion'):
    os.system('git clone https://github.com/levihsu/OOTDiffusion.git /kaggle/working/OOTDiffusion')

FLAG = '/kaggle/working/setup_done_v3.flag'
if not os.path.exists(FLAG):
    print('Installing packages...')
    os.system(f'{sys.executable} -m pip install -q -U "numpy<2.0.0" "scipy<1.13.0"')
    os.system(f'{sys.executable} -m pip install -q -U diffusers==0.24.0 transformers==4.36.2 huggingface_hub==0.22.2 einops onnxruntime accelerate "gradio==3.41.2" peft==0.8.2 datasets==2.16.1 sentence-transformers==2.3.1')
    open(FLAG, 'w').close()
    print('Done! Restarting kernel...')
    os._exit(0)
else:
    print('Already installed. Proceed to Step 2!')


### Step 2: Download Model Checkpoints
Run after the kernel restarts. Downloads weights from HuggingFace (~15 GB).

In [ ]:
from huggingface_hub import snapshot_download

print('Downloading OOTDiffusion checkpoints...')
snapshot_download(
    repo_id='levihsu/OOTDiffusion',
    local_dir='/kaggle/working/OOTDiffusion',
    ignore_patterns=['.git/*']
)
print('Downloading CLIP model...')
snapshot_download(
    repo_id='openai/clip-vit-large-patch14',
    local_dir='/kaggle/working/OOTDiffusion/checkpoints/clip-vit-large-patch14',
    ignore_patterns=['*.msgpack', '*.h5', '*.coreml', 'pytorch_model.bin']
)
print('All downloads complete!')


### Step 3: Launch Gradio UI (Best Quality Mode)
Runs at full 768x1024 resolution, 30 diffusion steps, lossless PNG pipeline.
A public `gradio.live` URL will be printed. Click it to open.

In [ ]:
import sys, os, gc, traceback
import gradio as gr
import torch
import numpy as np
from PIL import Image

repo_path = '/kaggle/working/OOTDiffusion'

# ── Path setup (identical to working copy notebook) ──
for p in [
    repo_path,
    f'{repo_path}/run',
    f'{repo_path}/preprocess',
    f'{repo_path}/preprocess/openpose',
    f'{repo_path}/preprocess/humanparsing',
]:
    if p not in sys.path:
        sys.path.insert(0, p)

open(f'{repo_path}/preprocess/openpose/config.py', 'w').close()
os.chdir(f'{repo_path}/run')

from utils_ootd import get_mask_location
from preprocess.openpose.run_openpose import OpenPose
from preprocess.humanparsing.run_parsing import Parsing
from ootd.inference_ootd_hd import OOTDiffusionHD

print('Loading AI models (takes 1-2 min on first run)...')
openpose_model = OpenPose(0)
parsing_model  = Parsing(0)
model          = OOTDiffusionHD(0)
print('Models loaded successfully!')

# Use PNG for zero quality loss (JPG adds compression artifacts)
TMP_GARMENT = '/kaggle/working/tmp_garment.png'
TMP_MODEL   = '/kaggle/working/tmp_model.png'

def run_tryon(garment_np, model_np, num_steps, image_scale, seed):
    try:
        # ── Clean GPU memory before every generation ──
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ── Save uploads as lossless PNG, then reload with Image.open() ──
        # This matches exactly how the working copy notebook loads images
        Image.fromarray(garment_np.astype(np.uint8)).save(TMP_GARMENT)
        Image.fromarray(model_np.astype(np.uint8)).save(TMP_MODEL)

        # ── Full resolution pipeline (768x1024) — same as copy notebook ──
        cloth_img = Image.open(TMP_GARMENT).convert('RGB').resize((768, 1024))
        model_img = Image.open(TMP_MODEL).convert('RGB').resize((768, 1024))

        # ── Preprocessing at native resolution (384x512) ──
        with torch.no_grad():
            keypoints      = openpose_model(model_img.resize((384, 512)))
            model_parse, _ = parsing_model(model_img.resize((384, 512)))

        mask, mask_gray = get_mask_location('hd', 'upper_body', model_parse, keypoints)
        mask      = mask.resize((768, 1024), Image.NEAREST)
        mask_gray = mask_gray.resize((768, 1024), Image.NEAREST)
        masked_vton_img = Image.composite(mask_gray, model_img, mask)

        # ── Inference with torch.no_grad for clean memory usage ──
        with torch.no_grad():
            images = model(
                model_type='hd',
                category='upperbody',
                image_garm=cloth_img,
                image_vton=masked_vton_img,
                mask=mask,
                image_ori=model_img,
                num_samples=1,
                num_steps=int(num_steps),
                image_scale=float(image_scale),
                seed=int(seed),
            )

        return images[0], ''

    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            gc.collect()
            torch.cuda.empty_cache()
            return None, 'GPU ran out of memory. Please click Generate again — memory has been freed.'
        return None, traceback.format_exc()
    except Exception:
        return None, traceback.format_exc()

with gr.Blocks(title='OOTDiffusion Virtual Try-On') as demo:
    gr.Markdown('# OOTDiffusion: Virtual Try-On')
    gr.Markdown('Upload a **garment** and a **person photo**, then click **Generate**. Upper-body garments only.')
    with gr.Row():
        garment_in = gr.Image(label='Garment Image', type='numpy')
        model_in   = gr.Image(label='Person Photo',  type='numpy')
        output_img = gr.Image(label='Try-On Result', type='pil')
    with gr.Row():
        steps_slider = gr.Slider(10, 50, value=30, step=1,   label='Steps (30 = best quality, 20 = faster)')
        scale_slider = gr.Slider(1.0, 5.0, value=2.0, step=0.1, label='Guidance Scale (2.0 recommended)')
        seed_input   = gr.Number(value=-1, label='Seed (-1 = random)')
    generate_btn = gr.Button('Generate Try-On', variant='primary')
    error_box    = gr.Textbox(label='Error Log', interactive=False, visible=True)
    generate_btn.click(
        fn=run_tryon,
        inputs=[garment_in, model_in, steps_slider, scale_slider, seed_input],
        outputs=[output_img, error_box],
    )

os.system('fuser -k 7860/tcp 2>/dev/null || true')
demo.launch(share=True)
